<div style="display: flex; align-items: center;">
    <h1>The AgMIP calibration protocol with diffWOFOST</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>

The AgMIP crop-model calibration workgroup (Wallach, Palosuo, Thorburn, Seidel, Mielenz, Buis) asked whether a differentiable WOFOST could carry out their protocol. This notebook does the part that is already well posed, on the dataset they actually circulate.

Their protocol is Wallach et al. (2024), *A calibration protocol for soil-crop models*, Environmental Modelling & Software, and the field test is Wallach et al. (2025), European Journal of Agronomy. Steps 1–5 are model expertise and produce tables. Steps 6–8 are calculations: ordinary least squares one variable group at a time, forward selection of candidate parameters with AICc, then one weighted least-squares refit of every selected parameter. The reference implementation is `CroptimizR::run_protocol_agmip`, which minimises with multi-start Nelder–Mead. The paper is explicit that the simplex is not part of the protocol. Any optimiser is allowed. diffWOFOST replaces it with Adam, because the stage dates are a smooth function of the parameters.

The WOFOST order used in CGMS training (phenology, then light interception and biomass, then partitioning, then water) is the same idea as their variable-group order. Phenology comes first in both, because later groups barely feed back onto it. This notebook runs that first group with the phenology model, then biomass, plant nitrogen, yield and grain protein with WOFOST 8.1 and SNOMIN. Both are on main.


## Dataset

Phase IV uses one French winter-wheat cultivar, Apache, from Arvalis variety trials. Twenty-two site-years are split so that calibration and evaluation share neither sites nor years. Fourteen site-years are for calibration. The public files are in [sbuis/AgMIP-Calibration-Phase-IV](https://github.com/sbuis/AgMIP-Calibration-Phase-IV):

- `data/cal_4_obs_French_A.txt`, the calibration observations
- `data/cal_4_obs_French_units.csv`, the units and their proposed groups
- `data/cal_4_input_data_French_A.txt`, soil, sowing, fertiliser and irrigation for all 22 site-years
- `data/cal_4_weather.xlsx`, daily weather for seven stations

The input file has all 22 site-years: weather can be joined from the station sheets, and soil, sowing and fertiliser are in that table. The observation file has only the 14 calibration site-years. The other eight were measured and are the evaluation set in Wallach et al. (2024, 2025). Those measurements are not in the public repository. Participants simulate them and send the simulations back; the organisers score them. This notebook cannot score them.

Emergence is a different gap. The units table defines a column `Date_BBCH10`, and the observation table has no such column. There is no emergence date to fit, in the calibration set or elsewhere in the download.

There is a second, synthetic copy of the same environments, generated with STICS, in [sbuis/AgMIP_calibration_PhaseIV_step2_synthetic_experiment](https://github.com/sbuis/AgMIP_calibration_PhaseIV_step2_synthetic_experiment). That set is for testing a protocol when the true parameters are known. It is not what a WOFOST entry would be judged on.


In [ ]:
import datetime as dt
import io
import tempfile
import urllib.request
import warnings
from pathlib import Path

import pandas as pd
import torch
from pcse.input import YAMLAgroManagementReader
from pcse.input import YAMLCropDataProvider
from pcse.models import Wofost81_NWLP_MLWB_SNOMIN

from diffwofost.physical_models.config import ComputeConfig
from diffwofost.physical_models.config import Configuration
from diffwofost.physical_models.crop.phenology import DVS_Phenology
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.parameter_providers import ParameterProvider

warnings.filterwarnings("ignore", message="Converting a tensor with requires_grad")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

DATA_URL = "https://raw.githubusercontent.com/sbuis/AgMIP-Calibration-Phase-IV/main/data"


def crop_directory():
    """Winter-wheat parameter file shipped with the SNOMIN notebook."""
    here = Path.cwd().resolve()
    relative = (
        Path("docs/notebooks/snomin_calibration_data/crop"),
        Path("snomin_calibration_data/crop"),
        Path("crop"),
    )
    for base in (here, *here.parents):
        for name in relative:
            candidate = base / name
            if (candidate / "winterwheat.yaml").exists():
                return candidate
    raise FileNotFoundError("winterwheat.yaml is not next to this notebook")


def download(name):
    with urllib.request.urlopen(f"{DATA_URL}/{name}") as response:
        return response.read()


observations = pd.read_csv(io.BytesIO(download("cal_4_obs_French_A.txt")), sep=r"\s+")
management = pd.read_csv(
    io.BytesIO(download("cal_4_input_data_French_A.txt")), sep=r"\s+", encoding="utf-8-sig"
)
units = pd.read_csv(io.BytesIO(download("cal_4_obs_French_units.csv")), sep=";", encoding="cp1252")
weather_book = pd.ExcelFile(io.BytesIO(download("cal_4_weather.xlsx")))
print(f"Calibration rows: {len(observations)}. Site-years in the input file: {len(management)}.")
print(f"Weather stations: {', '.join(weather_book.sheet_names)}.")
units


## Steps 1–5, filled in for WOFOST

Step 1. Defaults are cultivar Arminda in the WOFOST 8.1 winter-wheat file. That is the cultivar on which the 8.1 nitrogen parameters were estimated (Berghuijs et al., 2024). Apache is a French soft winter wheat. The protocol says to start from a cultivar of the same maturity class and to estimate the parameters that absorb the difference. Arminda uses vernalization (`IDSL = 2`).

Step 2 and 3. A measured variable enters a group only when WOFOST has a simulated equivalent. BBCH55 (heading) is taken as anthesis, development stage 1. Heading is a few days earlier than anthesis, so the estimated `TSUM1` is slightly short of a true anthesis requirement. BBCH90 is maturity, development stage 2. BBCH30 has no development-stage switch of its own: the time from emergence to heading is a single temperature sum. Ears and grain number are not state variables.

Step 4. Major parameters are few, and each one has to move the simulated variable in every environment. One thermal time per observed stage.

Step 5. Candidates are only kept when they reduce AICc. Vernalization first, because these are autumn-sown crops and the sites differ in winter temperature. Daylength last: the sites only span about 47–50°N.


In [ ]:
variables = pd.DataFrame(
    [
        ["Date_BBCH30", "none", "phenology", "", "Stem elongation is not a WOFOST stage."],
        ["Date_BBCH55", "day when DVS reaches 1", "phenology", 1, "Heading, used as anthesis."],
        ["Date_BBCH90", "day when DVS reaches 2", "phenology", 1, "Maturity."],
        ["Biomass", "TAGP", "plant_biomass", 2, "g m-2 in the file, kg ha-1 in WOFOST."],
        ["N_in_biomassHarvest", "crop N / TAGP", "plant_N_content", 3, "Percent. Needs SNOMIN."],
        ["Grain_Yield", "TWSO", "yield", 4, "g m-2 in the file, kg ha-1 in WOFOST."],
        ["ProteinContentGrain", "storage-organ N / TWSO", "seed_protein", 5, "Percent."],
        ["EarsPerSqM", "none", "", "", "WOFOST does not simulate ear number."],
        ["Grain_Number", "none", "", "", "WOFOST does not simulate grain number."],
    ],
    columns=["Observed", "Simulated", "Group", "Order", "Note"],
)
parameters = pd.DataFrame(
    [
        ["phenology", "major", "TSUM1", "Temperature sum, emergence to anthesis. Moves BBCH55."],
        ["phenology", "major", "TSUM2", "Temperature sum, anthesis to maturity. Moves BBCH90."],
        ["phenology", "candidate", "VERNSAT", "Saturated vernalization requirement."],
        ["phenology", "candidate", "VERNBASE", "Base vernalization requirement."],
        ["phenology", "candidate", "DLO", "Optimum daylength."],
        ["plant_biomass", "major", "AMAX_REF", "Assimilation level. One of the two allowed majors."],
        ["plant_biomass", "major", "SPAN", "Leaf lifespan. Sets how long biomass keeps accumulating."],
        ["plant_biomass", "candidate", "TDWI", "Initial biomass. Early-season residual."],
        ["plant_biomass", "candidate", "RGRLAI", "Juvenile leaf-area growth."],
        ["plant_N_content", "major", "RNUPTAKEMAX", "Maximum daily nitrogen uptake."],
        ["plant_N_content", "candidate", "NMAXSO", "Maximum nitrogen concentration in the grain."],
        ["yield", "major", "REALLOC_STEM_FRACTION", "Stem biomass that can move into the grain."],
        ["yield", "candidate", "CVO", "Conversion efficiency into storage organs."],
        ["seed_protein", "major", "NMAXSO", "Same parameter as above, if it was not already selected."],
    ],
    columns=["Group", "Role", "Parameter", "Why this one"],
)
display(variables)
display(parameters)


### Sum of squares

AICc treats the errors inside one group as independent draws from a single normal distribution. The raw sum of squares is an absolute measure: a miss of 50 g m-2 adds 2500 whether biomass is 100 or 2000 g m-2. In these trials biomass runs from about 100 g m-2 in early spring to about 2000 g m-2 at harvest, and the absolute miss tends to grow with it. A 20% miss is 20 g m-2 in March and 400 g m-2 in July. After squaring, the July points dominate and the early measurements barely move the fit. The logarithm turns the residual into a relative one, log(observed) minus log(simulated) is approximately the relative error, so a 20% miss counts about the same at both dates. The same transform applies to any other variable whose absolute scatter grows across the season.

Phenology stays in days. A miss of three days in April and a miss of three days in July are the same kind of error, so there is nothing to stabilise. Yield, grain protein and harvest nitrogen are one number per environment. They stay on the scale in the file: g m-2 for yield, percent for protein and for nitrogen in the biomass.

### Fertiliser

The input file records a date, a product and a nitrogen rate in kg N ha-1. SNOMIN does not take a nitrogen rate. Each dressing is an amendment: an amount of material, a depth, an organic-matter fraction, a C:N ratio, an apparent age, and the ammonium and nitrate fractions of that material. For these products the organic fraction is zero, so the C:N ratio and the age do nothing. Enter the file's nitrogen rate as `amount`. The ammonium and nitrate fractions then have to add up to one, or the model will apply a different quantity of nitrogen than the trial did.

Ammonitrate is ammonium nitrate, so half of the nitrogen is ammonium and half is nitrate (`f_NH4N = 0.5`, `f_NO3N = 0.5`). The product `18_46` is diammonium phosphate, and its nitrogen is ammonium (`f_NH4N = 1`, `f_NO3N = 0`). Solution azotée is a urea–ammonium nitrate solution, about half urea, a quarter ammonium and a quarter nitrate. SNOMIN has ammonium and nitrate pools and no urea hydrolysis. Urea becomes ammonium within a few days, so the urea share is added to the ammonium fraction (`f_NH4N = 0.75`, `f_NO3N = 0.25`). That drops the short delay before hydrolysis and any ammonia volatilisation.

The file gives no incorporation depth. Ten centimetres is the usual assumption. SNOMIN will not place an amendment above the first soil layer, and the top layer in these profiles is 25–30 cm, so the dressing is mixed through that layer.

### Soil

Each horizon has thickness, field capacity, wilting point, bulk density, texture, stones, pH and organic matter. Those are enough for the layered water balance, with a pedotransfer from texture to the retention curve, using the two measured water limits as checks. SNOMIN also needs `CNRatioSOMI`. The file has organic matter and no nitrogen or C:N, so every layer would keep one arable value, around 10. That number sets how much of the old organic matter is nitrogen, and therefore the background mineralisation. It is the same in every environment. Fitting it per site would absorb the yield residual and call it soil. The apparent age of that old organic matter is missing too. The SoilGrids example uses 24 years, and that is the value to keep fixed.

### What is fitted here

Phenology does not read the soil, so the loop below uses `DVS_Phenology` only. It is the group the protocol estimates first and then holds fixed. Biomass, plant nitrogen, yield and grain protein are estimated after that, with crop `Wofost81` and soil `SoilModuleWrapper_NWLP_MLWB_SNOMIN`.

### Why `TSUMEM` stays at its default

`TSUMEM` is the temperature sum from sowing to emergence. Development stage 0 is emergence, and `TSUM1` starts only then. BBCH10 is named in the units file and is absent from the observations. With only heading and maturity, raising `TSUMEM` and lowering `TSUM1` by the same amount moves both dates together. The two parameters are not separable, so `TSUMEM` stays at the Arminda value, 120 °C day.


## Steps 6–8 for phenology

The sum of squares is over heading and maturity, in every calibration environment. That is n = 28, two dates in each of the 14 calibration site-years. For independent Gaussian errors a fit enters the likelihood through n ln(SS/n). A smaller sum of squares is a better description of these dates. AIC adds a charge of 2 for every parameter that was adjusted,

$$
\mathrm{AIC} = n \ln(\mathrm{SS}/n) + 2p.
$$

The sum of squares in that formula is the minimum, after the freed parameters have been moved away from the starting cultivar. Arminda supplies the start. TSUM1 and TSUM2 are estimated because these heading and maturity dates are Apache's. Re-minimizing with one extra free parameter then gives a sum of squares lower than or equal to that minimum. It stays equal when the best value of the new parameter is the default already in use: the parameter does not move heading or maturity in these environments, or it does move them and the default was already the value the dates require. The charge of 2 is the smallest further drop in n ln(SS/n) that pays for one more parameter.

The protocol uses the small-sample version, AICc, because n is a few dozen:

$$
\mathrm{AICc} = n \ln(\mathrm{SS}/n) + 2p + \frac{2p(p+1)}{n-p-1}.
$$

With n = 28 the last term is 0.48 for the two major parameters and 1.0 once a third parameter is added. A candidate joins the estimated set only when AICc falls, which happens when the drop in n ln(SS/n) is larger than the rise in the penalty. For the two majors the sum of squares is 2063 and AICc is 124.9. Adding VERNSAT brings the sum of squares to about 1075 and AICc to 109.2, so it is kept. The likelihood term falls by about 18, and the penalty rises by about 2.5. VERNBASE and DLO raise AICc, to 114.2 and 116.0, and stay at the Arminda defaults.

The variance inside the logarithm is SS/n. The error variance printed after the selection is SS/(n-p). That number is the weight this group would carry once biomass, nitrogen and yield are in the same objective. With a single variable group, weighted least squares has the same minimiser as ordinary least squares, so step 7 does not move the parameters.

The date of a stage is a smooth step through the development trajectory, so the day itself has a gradient. Reported errors use the first day the stage is actually crossed.


In [ ]:
SHARPNESS = 40.0
BOUNDS = {
    "TSUM1": (400.0, 1400.0),
    "TSUM2": (400.0, 1400.0),
    "VERNSAT": (30.0, 90.0),
    "VERNBASE": (0.0, 20.0),
    "DLO": (12.0, 20.0),
}
MAJOR = ("TSUM1", "TSUM2")
CANDIDATES = ("VERNSAT", "VERNBASE", "DLO")
SCALARS = (
    "TSUMEM", "TBASEM", "TEFFMX", "TSUM1", "TSUM2", "IDSL", "DLO", "DLC",
    "DVSI", "DVSEND", "VERNSAT", "VERNBASE", "VERNDVS",
)


def parse_date(value):
    if pd.isna(value) or value == "NA":
        return None
    return dt.datetime.strptime(str(value), "%d/%m/%Y").date()


def station_weather(station):
    """Daily WOFOST weather. Radiation in the workbook is MJ m-2 day-1."""
    sheet = pd.read_excel(weather_book, f"station_{station}", header=None)
    latitude = float(str(sheet.iloc[0, -1]).split("Latitude =")[1].split("§")[0])
    altitude = float(str(sheet.iloc[0, -1]).split("Altitude =")[1])
    daily = sheet.iloc[5:].copy()
    daily.columns = [
        "DATECODE", "YYYY", "MM", "DD", "SRAD", "TMAX", "TMIN", "RAIN",
        "ET0", "WIND", "DEWP", "VPRS", "RHUM", "DAY",
    ]
    daily["DAY"] = pd.to_datetime(daily["DAY"]).dt.date
    return daily, latitude, altitude


def driving_rows(daily, latitude, altitude, start, end):
    window = daily[(daily["DAY"] >= start) & (daily["DAY"] <= end)]
    rows = []
    for record in window.itertuples(index=False):
        tmin = float(record.TMIN)
        tmax = float(record.TMAX)
        rows.append({
            "DAY": record.DAY,
            "LAT": latitude,
            "LON": 2.0,
            "ELEV": altitude,
            "IRRAD": float(record.SRAD) * 1.0e6,
            "TMIN": tmin,
            "TMAX": tmax,
            "TEMP": 0.5 * (tmin + tmax),
            "RAIN": float(record.RAIN) / 10.0,
            "VAP": 10.0,
            "WIND": 2.0,
        })
    return rows


def agro_calendar(start, end):
    text = f"""
AgroManagement:
- {start.isoformat()}:
    CropCalendar:
        crop_name: winterwheat
        variety_name: Arminda
        crop_start_date: {start.isoformat()}
        crop_start_type: sowing
        crop_end_date: {end.isoformat()}
        crop_end_type: harvest
        max_duration: 360
    TimedEvents: null
    StateEvents: null
"""
    with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as handle:
        handle.write(text)
        path = Path(handle.name)
    try:
        return YAMLAgroManagementReader(str(path))
    finally:
        path.unlink(missing_ok=True)


phenology_rows = (
    observations.dropna(subset=["Date_BBCH55", "Date_BBCH90"])
    .groupby("Number", as_index=False)
    .tail(1)
)
environments = []
for record in phenology_rows.itertuples(index=False):
    site = management.loc[management["Number"] == record.Number].iloc[0]
    sowing = parse_date(record.SowingDate)
    heading = parse_date(record.Date_BBCH55)
    maturity = parse_date(record.Date_BBCH90)
    harvest = parse_date(record.HarvestDate) or maturity
    daily, latitude, altitude = station_weather(int(site.Station))
    end = min(harvest + dt.timedelta(days=45), daily["DAY"].max())
    rows = driving_rows(daily, latitude, altitude, sowing, end)
    if not rows or rows[-1]["DAY"] < maturity:
        print(f"Skipping environment {record.Number}: weather does not cover maturity.")
        continue
    environments.append({
        "number": int(record.Number),
        "site": record.Site,
        "year": int(record.HarvestYear),
        "rows": rows,
        "agro": agro_calendar(sowing, rows[-1]["DAY"]),
        "heading": (heading - sowing).days,
        "maturity": (maturity - sowing).days,
    })

held_out = sorted(set(management["Number"]) - set(phenology_rows["Number"]))
print(f"Calibration environments with heading and maturity: {len(environments)}.")
print(f"Held-out environment numbers, observations not public: {held_out}.")

crop = YAMLCropDataProvider(
    Wofost81_NWLP_MLWB_SNOMIN, fpath=str(crop_directory()), force_reload=True
)
provider = ParameterProvider(cropdata=crop)
provider.set_active_crop("winterwheat", "Arminda", "sowing", "harvest")
for name in list(SCALARS) + ["DTSMTB", "VERNRTB"]:
    value = provider[name]
    if not torch.is_tensor(value):
        value = torch.tensor(value, dtype=torch.float64)
    provider.set_override(name, value, check=False)
defaults = {name: float(provider[name]) for name in list(BOUNDS) + ["TSUMEM", "IDSL"]}
print(
    "Arminda defaults: "
    + ", ".join(f"{name} {defaults[name]:.1f}" for name in ("TSUM1", "TSUM2", "VERNSAT", "VERNBASE", "DLO"))
)

config = Configuration(CROP=DVS_Phenology, OUTPUT_VARS=["DVS"])


In [ ]:
class BoundedParameter(torch.nn.Module):
    """A parameter kept inside its protocol bounds."""

    def __init__(self, low, high, init_value):
        super().__init__()
        self.low = low
        self.high = high
        fraction = (init_value - low) / (high - low)
        fraction = min(max(fraction, 1.0e-4), 1.0 - 1.0e-4)
        self.raw = torch.nn.Parameter(torch.logit(torch.tensor(fraction, dtype=torch.float64)))

    def forward(self):
        return self.low + (self.high - self.low) * torch.sigmoid(self.raw)


class ParameterSet(torch.nn.Module):
    def __init__(self, names, start):
        super().__init__()
        self.boxes = torch.nn.ModuleDict({
            name: BoundedParameter(*BOUNDS[name], start[name]) for name in names
        })

    def forward(self):
        return {name: box() for name, box in self.boxes.items()}


def apply(active):
    for name in SCALARS:
        if name in active:
            provider.set_override(name, active[name], check=False)
        elif name in accepted:
            provider.set_override(name, torch.tensor(accepted[name], dtype=torch.float64), check=False)


def development(environment, active):
    apply(active)
    engine = Engine(config)
    engine.setup(provider, iter(environment["rows"]), environment["agro"])
    engine.run_till(environment["rows"][-1]["DAY"])
    return torch.stack([day["DVS"].reshape(()) for day in engine.get_output()])


def stage_error(dvs, threshold, observed_day):
    """Error in days. A stage past the season end uses the remaining development."""
    past = dvs > threshold + 0.05
    capped = torch.where(past, dvs.detach(), dvs)
    previous = torch.cat((capped.new_zeros(1), capped[:-1]))
    entered = torch.sigmoid(SHARPNESS * (capped - threshold))
    entered = entered - torch.sigmoid(SHARPNESS * (previous - threshold))
    entered = entered.clamp(min=0.0)
    days = torch.arange(capped.shape[0], dtype=capped.dtype)
    weight = entered.sum()
    day = (entered * days).sum() / weight.clamp(min=1.0e-6)
    calendar = day - observed_day
    unfinished = (threshold - dvs[-1]).clamp(min=0.0) / 0.02
    beyond = unfinished + (dvs.shape[0] - 1 - observed_day)
    return torch.where(weight > 0.2, calendar, beyond)


def sum_of_squares(active):
    total = torch.zeros((), dtype=torch.float64)
    for environment in environments:
        dvs = development(environment, active)
        heading = stage_error(dvs, 1.0, environment["heading"])
        maturity = stage_error(dvs, 2.0, environment["maturity"])
        total = total + heading.square() + maturity.square()
    return total


def aicc(ss, n_obs, n_param):
    return n_obs * torch.log(ss / n_obs) + 2 * n_param + (2 * n_param * (n_param + 1)) / (n_obs - n_param - 1)


def estimate(names, start, steps):
    pack = ParameterSet(names, start)
    optimizer = torch.optim.Adam(pack.parameters(), lr=0.25)
    last = None
    for _ in range(steps):
        optimizer.zero_grad()
        last = sum_of_squares(pack())
        last.backward()
        optimizer.step()
    with torch.no_grad():
        values = {name: float(value) for name, value in pack().items()}
        ss = float(sum_of_squares(pack()))
    return values, ss


def hard_day(dvs, threshold):
    crossed = (dvs >= threshold).nonzero()
    if len(crossed) == 0:
        return None
    return int(crossed[0])


def stage_errors(values):
    """Signed errors in days, heading then maturity, one pair per environment."""
    heading, maturity = [], []
    with torch.no_grad():
        active = {name: torch.tensor(values[name], dtype=torch.float64) for name in values}
        for environment in environments:
            dvs = development(environment, active)
            for threshold, observed, store in (
                (1.0, environment["heading"], heading),
                (2.0, environment["maturity"], maturity),
            ):
                day = hard_day(dvs, threshold)
                store.append(None if day is None else day - observed)
    return heading, maturity


n_observations = 2 * len(environments)
accepted = {name: defaults[name] for name in BOUNDS}
history = []

major_values, major_ss = estimate(MAJOR, accepted, steps=10)
accepted.update(major_values)
selected = list(MAJOR)
best_aicc = float(aicc(torch.tensor(major_ss), n_observations, len(selected)))
history.append({"Parameters": ", ".join(selected), "SS": major_ss, "AICc": best_aicc, "Kept": "major"})
print(f"Major parameters. SS {major_ss:.1f}. AICc {best_aicc:.2f}.")

for candidate in CANDIDATES:
    trial = selected + [candidate]
    values, ss = estimate(trial, accepted, steps=8)
    score = float(aicc(torch.tensor(ss), n_observations, len(trial)))
    keep = score < best_aicc
    history.append({"Parameters": ", ".join(trial), "SS": ss, "AICc": score, "Kept": "yes" if keep else "no"})
    print(f"{candidate}: AICc {score:.2f} against {best_aicc:.2f}. {'Kept' if keep else 'Left at the default'}.")
    if keep:
        accepted.update(values)
        selected = trial
        best_aicc = score

final_ss = next(row["SS"] for row in reversed(history) if row["Kept"] != "no")
error_variance = final_ss / (n_observations - len(selected))
print(
    f"Step 7 weight for phenology, SS/(n-p) after step 6: {error_variance:.2f} day². "
    "One group makes weighted least squares the same fit."
)
pd.DataFrame(history)


In [ ]:
def summarize(errors):
    series = pd.Series([error for error in errors if error is not None], dtype=float)
    if series.empty:
        return {"n": 0, "bias": None, "RMSE": None}
    return {
        "n": int(series.size),
        "bias": series.mean(),
        "RMSE": (series.pow(2).mean()) ** 0.5,
    }


default_heading, default_maturity = stage_errors({name: defaults[name] for name in BOUNDS})
final_heading, final_maturity = stage_errors(accepted)
metrics = pd.DataFrame(
    [
        {"Stage": "BBCH55", "Parameters": "Arminda default", **summarize(default_heading)},
        {"Stage": "BBCH55", "Parameters": "after step 6", **summarize(final_heading)},
        {"Stage": "BBCH90", "Parameters": "Arminda default", **summarize(default_maturity)},
        {"Stage": "BBCH90", "Parameters": "after step 6", **summarize(final_maturity)},
    ]
)
estimated = pd.DataFrame(
    {
        "Parameter": list(BOUNDS),
        "Role": ["selected" if name in selected else "default" for name in BOUNDS],
        "Default": [defaults[name] for name in BOUNDS],
        "Estimated": [accepted[name] for name in BOUNDS],
    }
)
display(metrics.round(2))
display(estimated.round(2))


## Steps 6 and 7 for biomass, nitrogen, yield and protein

Phenology stays at the values selected above while these groups are estimated. Biomass uses every biomass date. The file is in g m-2 and `TAGP` is in kg ha-1, so each observation is multiplied by 10 and then log-transformed. Yield stays in g m-2 and is compared with `TWSO` / 10. Plant nitrogen is the harvest percentage of `TAGP` that is still in leaves, stems and grain. Grain protein is 5.7 times the nitrogen percentage of `TWSO`, the factor used for wheat. A protein of zero in the file is a missing value.

The soil curve is the Wösten pedotransfer from clay, silt, organic matter and bulk density. Field capacity and wilting point in the file are the check on that curve. The available-water column matches those two limits and bulk density without a further reduction for stones, so the curve is not scaled by the stone percentage. Every layer keeps `CNRatioSOMI` at 10 and an apparent age of 24 years. The top layer starts with 2 kg NH4-N ha-1 and 10 kg NO3-N ha-1; the dressings in the file supply the rest. Rooting depth is the layer boundary nearest 125 cm, because `RDMCR` has to land on a boundary. A shallower profile uses its own depth.

One gradient step is a full season, about ten seconds, and each step runs all 14 environments, so the group search and the weighted refit together take about two hours. Majors get 4 Adam steps and each candidate gets 3. The refit of every selected parameter, including phenology, gets 4 steps. Each group is weighted by the inverse of its error variance, SS/(n-p).


In [ ]:
import math

from pcse.input import WOFOST81SiteDataProvider_SNOMIN

from diffwofost.physical_models.crop.wofost81 import Wofost81
from diffwofost.physical_models.soil.soil_wrappers import SoilModuleWrapper_NWLP_MLWB_SNOMIN

# A gradient step is one full season. Four steps for majors, three per candidate.
MAJOR_STEPS = 4
CANDIDATE_STEPS = 3
WLS_STEPS = 4
GROUP_LR = 0.12
PFS = (-1.0, 1.0, 1.3, 1.7, 2.0, 2.3, 2.4, 2.7, 3.0, 3.3, 3.7, 4.0, 4.2, 6.0)
CN_RATIO = 10.0
ROOT_TARGET = 125.0
PROTEIN_FACTOR = 5.7
GROWTH_BOUNDS = {
    "AMAX_REF": (20.0, 70.0),
    "SPAN": (15.0, 55.0),
    "TDWI": (50.0, 800.0),
    "RGRLAI": (0.003, 0.02),
    "RNUPTAKEMAX": (1.0, 8.0),
    "NMAXSO": (0.012, 0.030),
    "REALLOC_STEM_FRACTION": (0.05, 0.60),
    "CVO": (0.55, 0.85),
}
BOUNDS.update(GROWTH_BOUNDS)
GROUPS = (
    {"name": "plant_biomass", "major": ("AMAX_REF", "SPAN"), "candidates": ("TDWI", "RGRLAI"), "kind": "biomass"},
    {"name": "plant_N_content", "major": ("RNUPTAKEMAX",), "candidates": ("NMAXSO",), "kind": "nitrogen"},
    {"name": "yield", "major": ("REALLOC_STEM_FRACTION",), "candidates": ("CVO",), "kind": "yield"},
    {"name": "seed_protein", "major": ("NMAXSO",), "candidates": (), "kind": "protein"},
)
full_config = Configuration(
    CROP=Wofost81,
    SOIL=SoilModuleWrapper_NWLP_MLWB_SNOMIN,
    OUTPUT_VARS=["DVS", "TAGP", "TWSO", "NamountLV", "NamountST", "NamountSO"],
)


def wosten(clay, silt, organic, density, topsoil):
    """Van Genuchten parameters from the Wösten pedotransfer."""
    clay = max(float(clay), 1.0)
    silt = max(float(silt), 1.0)
    organic = max(float(organic), 0.01)
    density = float(density)
    top = 1.0 if topsoil else 0.0
    t_alpha = (
        -14.96 + 0.03135 * clay + 0.0351 * silt + 0.646 * organic + 15.29 * density
        - 0.192 * top - 4.671 * density**2 - 0.000781 * clay**2 - 0.00687 * organic**2
        + 0.0449 / organic + 0.0663 * math.log(silt) + 0.1482 * math.log(organic)
        - 0.04546 * density * silt - 0.4852 * density * organic + 0.00673 * top * clay
    )
    t_n = (
        -25.23 - 0.02195 * clay + 0.0074 * silt - 0.1940 * organic + 45.5 * density
        - 7.24 * density**2 + 0.0003658 * clay**2 + 0.002885 * organic**2 - 12.81 / density
        - 0.1524 / silt - 0.01958 / organic - 0.2876 * math.log(silt) - 0.0709 * math.log(organic)
        - 44.6 * math.log(density) - 0.02264 * density * clay + 0.0896 * density * organic
        + 0.00718 * top * clay
    )
    t_lambda = (
        0.0202 + 0.0006193 * clay**2 - 0.001136 * organic**2 - 0.2316 * math.log(organic)
        - 0.03544 * density * clay + 0.00283 * density * silt + 0.0488 * density * organic
    )
    t_ksat = (
        7.755 + 0.0352 * silt + 0.93 * top - 0.967 * density**2 - 0.000484 * clay**2
        - 0.000322 * silt**2 + 0.001 / silt - 0.0748 / organic - 0.643 * math.log(silt)
        - 0.01398 * density * clay - 0.1673 * density * organic + 0.02986 * top * clay
        - 0.03305 * top * silt
    )
    theta_s = (
        0.7919 + 0.001691 * clay - 0.29619 * density - 1.491e-6 * silt**2
        + 8.21e-5 * organic**2 + 0.02427 / clay + 0.01113 / silt + 0.01472 * math.log(silt)
        - 7.33e-5 * organic * clay - 0.000619 * density * clay - 0.001183 * density * organic
        - 0.0001664 * top * silt
    )
    return {
        "alpha": math.exp(t_alpha),
        "n": math.exp(t_n) + 1.0,
        "lam": (10 * math.exp(t_lambda) - 10) / (1 + math.exp(t_lambda)),
        "k_sat": math.exp(t_ksat),
        "theta_s": min(max(theta_s, 0.25), 0.65),
        "theta_r": 0.01,
    }


def _moisture(pf, vg):
    psi = 10.0 ** pf
    n = vg["n"]
    m = 1.0 - 1.0 / n
    return vg["theta_r"] + (vg["theta_s"] - vg["theta_r"]) / (1.0 + (vg["alpha"] * psi) ** n) ** m


def _log_k(pf, vg):
    psi = 10.0 ** pf
    n = vg["n"]
    m = 1.0 - 1.0 / n
    ah = vg["alpha"] * psi
    h1 = (1.0 + ah ** n) ** m
    h2 = ah ** (n - 1.0)
    denom = (1.0 + ah ** n) ** (m * (vg["lam"] + 2.0))
    return math.log10(max(vg["k_sat"] * (h1 - h2) ** 2 / denom, 1e-12))


def retention_tables(vg):
    moistures = [_moisture(pf, vg) for pf in PFS]
    for index in range(1, len(moistures)):
        if moistures[index] >= moistures[index - 1]:
            moistures[index] = moistures[index - 1] - 1e-4
    sm, cond = [], []
    for pf, value in zip(PFS, moistures, strict=True):
        sm.extend((float(pf), max(value, 0.005)))
        cond.extend((float(pf), _log_k(pf, vg)))
    return sm, cond


def cell_number(row, name):
    value = row[name]
    if pd.isna(value) or value == "NA":
        return None
    return float(value)


def soil_layers(row):
    layers = []
    for horizon in range(1, int(row.N_Horizons) + 1):
        thickness = cell_number(row, f"Thickness_H{horizon}")
        if thickness is None:
            continue
        clay = cell_number(row, f"Clay_H{horizon}") or 10.0
        silt = (cell_number(row, f"Fine_silt_H{horizon}") or 0.0) + (
            cell_number(row, f"Coarse_silt_H{horizon}") or 0.0
        )
        organic = cell_number(row, f"Organic_matter_H{horizon}")
        organic = 1.0 if organic is None else organic
        density = cell_number(row, f"Apparent_density_H{horizon}") or 1.4
        ph = cell_number(row, f"Water_pH_H{horizon}") or 7.0
        capacity = cell_number(row, f"Field_capacity_H{horizon}")
        wilting = cell_number(row, f"Wilting_point_H{horizon}")
        vg = wosten(clay, silt, organic, density, topsoil=(horizon == 1))
        sm, cond = retention_tables(vg)
        layers.append({
            "Thickness": thickness,
            "SMfromPF": sm,
            "CONDfromPF": cond,
            "CRAIRC": 0.06,
            "CNRatioSOMI": CN_RATIO,
            "RHOD": density,
            "Soil_pH": ph,
            "FSOMI": organic / 100.0,
            "file_fc": None if capacity is None else capacity / 100.0 * density,
            "ptf_fc": _moisture(2.0, vg),
            "file_wp": None if wilting is None else wilting / 100.0 * density,
            "ptf_wp": _moisture(4.2, vg),
        })
    return layers


def with_root_boundary(layers):
    """Split a layer so the rooting depth lands on a boundary at or nearest 125 cm."""
    total = sum(layer["Thickness"] for layer in layers)
    target = min(ROOT_TARGET, total)
    depth = 0.0
    split = []
    for layer in layers:
        thickness = layer["Thickness"]
        crosses = depth < target < depth + thickness - 0.05
        upper = target - depth
        lower = thickness - upper
        if crosses and upper >= 5.0 and lower >= 5.0:
            split.append({**layer, "Thickness": upper})
            split.append({**layer, "Thickness": lower})
        else:
            split.append(layer)
            if crosses:
                target = depth + thickness
        depth += thickness
    boundaries, depth = [], 0.0
    for layer in split:
        depth += layer["Thickness"]
        boundaries.append(depth)
    if not any(abs(boundary - target) < 0.05 for boundary in boundaries):
        target = min(boundaries, key=lambda boundary: abs(boundary - target))
    return split, target


def fertiliser_fractions(product):
    key = str(product).lower().replace("é", "e").replace("è", "e")
    if "ammonitrate" in key:
        return 0.5, 0.5
    if "18_46" in key or "18-46" in key:
        return 1.0, 0.0
    if "solution" in key:
        return 0.75, 0.25
    raise ValueError(f"Unknown fertiliser product: {product}")


def timed_block(title, signal, events):
    if not events:
        return ""
    lines = [
        "      - event_signal: " + signal,
        f"        name: {title}",
        "        comment: recorded in the AgMIP input file",
        "        events_table:",
    ]
    for day, body in events:
        lines.append(f"          - {day.isoformat()}:")
        lines.extend(f"              {key}: {value}" for key, value in body.items())
    return "\n".join(lines)


def season_agro(start, end, dressings, irrigations):
    timed = "\n".join(part for part in (
        timed_block("Fertiliser", "apply_n_snomin", dressings),
        timed_block("Irrigation", "irrigate", irrigations),
    ) if part)
    text = f"""
AgroManagement:
- {start.isoformat()}:
    CropCalendar:
        crop_name: winterwheat
        variety_name: Arminda
        crop_start_date: {start.isoformat()}
        crop_start_type: sowing
        crop_end_date: {end.isoformat()}
        crop_end_type: harvest
        max_duration: 360
    TimedEvents:
{timed or "      []"}
    StateEvents: null
"""
    with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as handle:
        handle.write(text)
        path = Path(handle.name)
    try:
        return YAMLAgroManagementReader(str(path))
    finally:
        path.unlink(missing_ok=True)


def dressing_events(row, start, end):
    events = []
    count = 0 if pd.isna(row.Number_of_fertilizer_doses) else int(row.Number_of_fertilizer_doses)
    for index in range(1, count + 1):
        day = parse_date(getattr(row, f"Date_fertilization_{index}"))
        amount = cell_number(row, f"Amount_fertilization_{index}")
        product = getattr(row, f"Product_fertilization_{index}")
        if day is None or amount is None or not (start <= day <= end):
            continue
        ammonium, nitrate = fertiliser_fractions(product)
        events.append((day, {
            "amount": amount,
            "application_depth": 10,
            "cnratio": 0,
            "initial_age": 0,
            "f_NH4N": ammonium,
            "f_NO3N": nitrate,
            "f_orgmat": 0,
        }))
    return events


def irrigation_events(row, start, end):
    events = []
    count = 0 if pd.isna(row.Number_of_irrigations) else int(row.Number_of_irrigations)
    for index in range(1, count + 1):
        day = parse_date(getattr(row, f"Date_irrigation_{index}"))
        amount = cell_number(row, f"Amount_irrigation_{index}")
        if day is None or amount is None or not (start <= day <= end):
            continue
        events.append((day, {"amount": amount / 10.0, "efficiency": 1.0}))
    return events


def season_rows(daily, latitude, altitude, start, end):
    window = daily[(daily["DAY"] >= start) & (daily["DAY"] <= end)]
    rows = []
    for record in window.itertuples(index=False):
        tmin = float(record.TMIN)
        tmax = float(record.TMAX)
        temp = 0.5 * (tmin + tmax)
        # The sheet stores ET0 in mm day-1. E0 and ES0 are not in the file.
        reference = float(record.ET0) / 10.0
        rows.append({
            "DAY": record.DAY,
            "LAT": latitude,
            "LON": 2.0,
            "ELEV": altitude,
            "IRRAD": float(record.SRAD) * 1.0e6,
            "TMIN": tmin,
            "TMAX": tmax,
            "TEMP": temp,
            "DTEMP": 0.5 * (temp + tmax),
            "RAIN": float(record.RAIN) / 10.0,
            "VAP": 10.0,
            "WIND": 2.0,
            "ET0": reference,
            "E0": reference,
            "ES0": reference,
        })
    return rows


def harvest_value(block, column, missing_zero=False):
    rows = block.dropna(subset=[column])
    if rows.empty:
        return None
    value = float(rows.iloc[-1][column])
    if missing_zero and value == 0.0:
        return None
    return value


def build_season(number, weather_cache):
    block = observations.loc[observations["Number"] == number]
    site = management.loc[management["Number"] == number].iloc[0]
    sowing = parse_date(block.iloc[0].SowingDate)
    harvest = parse_date(block.dropna(subset=["HarvestDate"]).iloc[-1].HarvestDate)
    station = int(site.Station)
    if station not in weather_cache:
        weather_cache[station] = station_weather(station)
    daily, latitude, altitude = weather_cache[station]
    end = min(harvest, daily["DAY"].max())
    layers, root_depth = with_root_boundary(soil_layers(site))
    clean = []
    for layer in layers:
        clean.append({key: value for key, value in layer.items() if key not in {"file_fc", "ptf_fc", "file_wp", "ptf_wp"}})
    description = {
        "PFWiltingPoint": 4.2,
        "PFFieldCapacity": 2.0,
        "SurfaceConductivity": 70.0,
        "SoilLayers": clean,
        "SubSoilType": {**clean[-1], "Thickness": 200.0, "FSOMI": 0.0},
        "GroundWater": None,
    }
    wav = cell_number(site, "Available_soil_water,_volumetric")
    wav = 20.0 if wav is None else min(wav / 10.0, 100.0)
    n_layers = len(clean)
    site_data = WOFOST81SiteDataProvider_SNOMIN(
        WAV=wav, CO2=400.0, A0SOM=24.0,
        NH4I=[2.0] + [0.0] * (n_layers - 1),
        NO3I=[10.0] + [0.0] * (n_layers - 1),
    )
    season_provider = ParameterProvider(
        cropdata=crop, soildata={"RDMSOL": root_depth, "SoilProfileDescription": description}, sitedata=site_data
    )
    season_provider.set_active_crop("winterwheat", "Arminda", "sowing", "harvest")
    biomass = []
    for record in block.itertuples(index=False):
        if pd.isna(record.Biomass):
            continue
        day = (parse_date(record.Date) - sowing).days
        biomass.append((day, float(record.Biomass) * 10.0))
    return {
        "number": int(number),
        "provider": season_provider,
        "rows": season_rows(daily, latitude, altitude, sowing, end),
        "agro": season_agro(sowing, end, dressing_events(site, sowing, end), irrigation_events(site, sowing, end)),
        "end": end,
        "rdmcr": root_depth,
        "biomass": biomass,
        "yield_gm2": harvest_value(block, "Grain_Yield"),
        "nitrogen": harvest_value(block, "N_in_biomassHarvest"),
        "protein": harvest_value(block, "ProteinContentGrain", missing_zero=True),
        "checks": [(layer["file_fc"], layer["ptf_fc"], layer["file_wp"], layer["ptf_wp"]) for layer in layers],
    }


weather_cache = {}
seasons = [build_season(number, weather_cache) for number in sorted(observations["Number"].unique())]
fc_gap, wp_gap = [], []
for season in seasons:
    for file_fc, ptf_fc, file_wp, ptf_wp in season["checks"]:
        if file_fc is not None:
            fc_gap.append(abs(file_fc - ptf_fc))
        if file_wp is not None:
            wp_gap.append(abs(file_wp - ptf_wp))
print(
    f"Seasons: {len(seasons)}. "
    f"Biomass dates: {sum(len(season['biomass']) for season in seasons)}. "
    f"Yield: {sum(season['yield_gm2'] is not None for season in seasons)}. "
    f"Plant N: {sum(season['nitrogen'] is not None for season in seasons)}. "
    f"Protein: {sum(season['protein'] is not None for season in seasons)}."
)
print(
    f"Pedotransfer versus the file, mean absolute volumetric gap: "
    f"field capacity {sum(fc_gap)/len(fc_gap):.3f}, wilting point {sum(wp_gap)/len(wp_gap):.3f}."
)

current = {name: float(accepted[name]) for name in accepted}
for name in GROWTH_BOUNDS:
    current[name] = float(provider[name])
arminda_values = {name: defaults[name] for name in ("TSUM1", "TSUM2", "VERNSAT", "VERNBASE", "DLO")}
arminda_values.update({name: float(provider[name]) for name in GROWTH_BOUNDS})


def merged(active):
    values = {name: torch.tensor(float(current[name]), dtype=torch.float64) for name in current}
    values.update(active)
    return values


def simulate(season, values):
    season_provider = season["provider"]
    for name, value in values.items():
        if name in BOUNDS:
            season_provider.set_override(name, value, check=False)
    season_provider.set_override("RDMCR", season["rdmcr"], check=False)
    engine = Engine(full_config)
    engine.setup(season_provider, iter(season["rows"]), season["agro"])
    engine.run_till(season["end"])
    output = engine.get_output()
    return {name: torch.stack([day[name].reshape(()) for day in output]) for name in full_config.OUTPUT_VARS}


def season_ss(season, values, kind):
    output = simulate(season, values)
    if kind == "biomass":
        total = output["TAGP"].new_zeros(())
        for day, observed in season["biomass"]:
            if day < 0 or day >= output["TAGP"].shape[0]:
                continue
            total = total + (torch.log(output["TAGP"][day].clamp(min=1.0)) - math.log(observed)).square()
        return total
    if kind == "yield":
        if season["yield_gm2"] is None:
            return output["TWSO"].new_zeros(())
        return (output["TWSO"][-1] / 10.0 - season["yield_gm2"]).square()
    if kind == "nitrogen":
        if season["nitrogen"] is None:
            return output["TAGP"].new_zeros(())
        crop_n = output["NamountLV"][-1] + output["NamountST"][-1] + output["NamountSO"][-1]
        return (100.0 * crop_n / output["TAGP"][-1].clamp(min=1.0) - season["nitrogen"]).square()
    if season["protein"] is None:
        return output["TWSO"].new_zeros(())
    predicted = 100.0 * PROTEIN_FACTOR * output["NamountSO"][-1] / output["TWSO"][-1].clamp(min=1.0)
    return (predicted - season["protein"]).square()


def observation_count(kind):
    if kind == "biomass":
        return sum(len(season["biomass"]) for season in seasons)
    key = {"yield": "yield_gm2", "nitrogen": "nitrogen", "protein": "protein"}[kind]
    return sum(season[key] is not None for season in seasons)


def evaluate_kind(kind, values):
    frozen = {name: torch.tensor(float(values[name]), dtype=torch.float64) for name in values}
    total = 0.0
    with torch.no_grad():
        for season in seasons:
            total += float(season_ss(season, frozen, kind))
    return total


def fit_kind(names, kind, steps):
    """Adam on `names`. Each step backpropagates one season at a time."""
    pack = ParameterSet(names, current)
    optimizer = torch.optim.Adam(pack.parameters(), lr=GROUP_LR)
    for step in range(steps):
        optimizer.zero_grad()
        total = 0.0
        for season in seasons:
            piece = season_ss(season, merged(pack()), kind)
            if piece.requires_grad:
                piece.backward()
                total += float(piece.detach())
        optimizer.step()
        text = ", ".join(f"{name} {float(pack()[name].detach()):.4g}" for name in names)
        print(f"  step {step + 1}/{steps} SS {total:.1f}. {text}")
    frozen = {name: float(value.detach()) for name, value in pack().items()}
    return frozen, evaluate_kind(kind, {**current, **frozen})


group_rows = []
group_variance = {"phenology": float(error_variance)}
for spec in GROUPS:
    kind = spec["kind"]
    n_obs = observation_count(kind)
    free_majors = [name for name in spec["major"] if name not in selected]
    free_candidates = [name for name in spec["candidates"] if name not in selected]
    print(f"{spec['name']}. n = {n_obs}. Majors: {', '.join(free_majors) or 'already selected'}.")
    chosen = []
    if free_majors:
        fitted, ss = fit_kind(free_majors, kind, MAJOR_STEPS)
        current.update(fitted)
        chosen = list(free_majors)
    else:
        ss = evaluate_kind(kind, current)
    best_ss = ss
    best_score = None
    if chosen:
        best_score = float(aicc(torch.tensor(ss), n_obs, len(chosen)))
        group_rows.append({
            "Group": spec["name"], "Parameters": ", ".join(chosen),
            "SS": ss, "AICc": best_score, "Kept": "major",
        })
        print(f"  Majors SS {ss:.1f}. AICc {best_score:.2f}.")
    for candidate in free_candidates:
        trial = chosen + [candidate]
        fitted, ss = fit_kind(trial, kind, CANDIDATE_STEPS)
        score = float(aicc(torch.tensor(ss), n_obs, len(trial)))
        keep = best_score is None or score < best_score
        group_rows.append({
            "Group": spec["name"], "Parameters": ", ".join(trial),
            "SS": ss, "AICc": score, "Kept": "yes" if keep else "no",
        })
        print(f"  {candidate}: AICc {score:.2f} against {best_score}. {'Kept' if keep else 'Left at the default'}.")
        if keep:
            current.update(fitted)
            chosen = trial
            best_ss = ss
            best_score = score
    if chosen:
        selected.extend(name for name in chosen if name not in selected)
    denom = n_obs - len(chosen)
    group_variance[kind] = best_ss / denom if denom > 0 else best_ss / n_obs
    print(f"  Weight SS/(n-p) = {group_variance[kind]:.4g}.")

print("Weighted refit of every selected parameter.")


def fit_weighted(steps):
    pack = ParameterSet(selected, current)
    optimizer = torch.optim.Adam(pack.parameters(), lr=GROUP_LR)
    for step in range(steps):
        optimizer.zero_grad()
        total = 0.0
        pheno = sum_of_squares(pack()) / group_variance["phenology"]
        pheno.backward()
        total += float(pheno.detach())
        for spec in GROUPS:
            variance = group_variance[spec["kind"]]
            for season in seasons:
                piece = season_ss(season, merged(pack()), spec["kind"]) / variance
                if piece.requires_grad:
                    piece.backward()
                    total += float(piece.detach())
        optimizer.step()
        print(f"  weighted step {step + 1}/{steps} objective {total:.2f}")
    return {name: float(value.detach()) for name, value in pack().items()}


after_step6 = dict(current)
current.update(fit_weighted(WLS_STEPS))
pd.DataFrame(group_rows)


## Step 8 on the calibration site-years

These 14 site-years are the observations the parameters were fitted to. The table gives bias, RMSE and RRMSE at the Arminda default, after the separate group fits, and after the weighted refit. Bias is simulated minus observed. RRMSE is the RMSE divided by the mean of the observations. Biomass is reported in g m-2 and as its logarithm, the scale used in that group's sum of squares. The figure shows the same site-years for the Arminda default and the final parameters. The eight evaluation environments stay out of this comparison: their observations are not in the public file.


In [ ]:
import matplotlib.pyplot as plt

STAGE = {"BBCH55": 1.0, "BBCH90": 2.0}
OBSERVED_STAGE = {
    environment["number"]: {"BBCH55": environment["heading"], "BBCH90": environment["maturity"]}
    for environment in environments
}
VARIABLES = ["BBCH55", "BBCH90", "biomass", "ln(biomass)", "plant N", "yield", "protein"]
PLOT_VARIABLES = ["BBCH55", "BBCH90", "biomass", "plant N", "yield", "protein"]
UNITS = {
    "BBCH55": "days after sowing",
    "BBCH90": "days after sowing",
    "biomass": "g m-2",
    "plant N": "% N",
    "yield": "g m-2",
    "protein": "% protein",
}


def calibration_pairs(values):
    """Observed and simulated values on the calibration site-years."""
    frozen = {name: torch.tensor(float(values[name]), dtype=torch.float64) for name in values}
    rows = []
    missed = []
    with torch.no_grad():
        for season in seasons:
            output = simulate(season, frozen)
            number = season["number"]
            for name, threshold in STAGE.items():
                day = hard_day(output["DVS"], threshold)
                observed = OBSERVED_STAGE[number][name]
                if day is None:
                    missed.append((number, name))
                    continue
                rows.append({"variable": name, "observed": observed, "simulated": day, "number": number})
            for day, observed_kg in season["biomass"]:
                if day < 0 or day >= output["TAGP"].shape[0]:
                    continue
                observed = observed_kg / 10.0
                simulated = float(output["TAGP"][day]) / 10.0
                rows.append({"variable": "biomass", "observed": observed, "simulated": simulated, "number": number})
                rows.append({
                    "variable": "ln(biomass)",
                    "observed": math.log(observed),
                    "simulated": math.log(max(simulated, 0.1)),
                    "number": number,
                })
            if season["yield_gm2"] is not None:
                rows.append({
                    "variable": "yield",
                    "observed": season["yield_gm2"],
                    "simulated": float(output["TWSO"][-1]) / 10.0,
                    "number": number,
                })
            if season["nitrogen"] is not None:
                crop_n = float(output["NamountLV"][-1] + output["NamountST"][-1] + output["NamountSO"][-1])
                predicted = 100.0 * crop_n / max(float(output["TAGP"][-1]), 1.0)
                rows.append({"variable": "plant N", "observed": season["nitrogen"], "simulated": predicted, "number": number})
            if season["protein"] is not None:
                predicted = 100.0 * PROTEIN_FACTOR * float(output["NamountSO"][-1]) / max(float(output["TWSO"][-1]), 1.0)
                rows.append({"variable": "protein", "observed": season["protein"], "simulated": predicted, "number": number})
    if missed:
        print("Stage not reached: " + ", ".join(f"{number} {name}" for number, name in missed))
    return pd.DataFrame(rows)


def agreement_rows(pairs, label):
    rows = []
    for variable in VARIABLES:
        block = pairs.loc[pairs["variable"] == variable]
        if block.empty:
            rows.append({"Variable": variable, "Parameters": label, "n": 0, "bias": None, "RMSE": None, "RRMSE": None})
            continue
        error = block["simulated"] - block["observed"]
        rmse = float((error.pow(2).mean()) ** 0.5)
        scale = float(block["observed"].abs().mean())
        rows.append({
            "Variable": variable,
            "Parameters": label,
            "n": int(len(block)),
            "bias": float(error.mean()),
            "RMSE": rmse,
            "RRMSE": rmse / scale,
        })
    return rows


parameter_sets = (
    ("Arminda default", arminda_values),
    ("after step 6", after_step6),
    ("after step 7", current),
)
calibration_points = {}
rows = []
for label, values in parameter_sets:
    print(f"Calibration agreement, {label}.")
    calibration_points[label] = calibration_pairs(values)
    rows.extend(agreement_rows(calibration_points[label], label))
agreement = pd.DataFrame(rows)
display(agreement.round(3))

fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
for ax, variable in zip(axes.ravel(), PLOT_VARIABLES):
    limits = []
    for label, marker in (("Arminda default", "o"), ("after step 7", "s")):
        block = calibration_points[label]
        block = block.loc[block["variable"] == variable]
        if block.empty:
            continue
        ax.scatter(block["observed"], block["simulated"], marker=marker, s=28, label=label)
        limits.extend(block["observed"].tolist())
        limits.extend(block["simulated"].tolist())
    if not limits:
        ax.set_title(variable)
        continue
    low, high = min(limits), max(limits)
    pad = 0.05 * (high - low or 1.0)
    ax.plot([low - pad, high + pad], [low - pad, high + pad], color="0.5", linewidth=1)
    ax.set_title(variable)
    ax.set_xlabel(f"observed, {UNITS[variable]}")
    ax.set_ylabel("simulated")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Calibration site-years")
fig.tight_layout()
plt.show()


## What this does and does not answer

Steps 6 and 7 are the group-wise least squares with AICc, then one weighted refit. The optimiser is the part that changes. Nelder–Mead in `CroptimizR` needs repeated starts because a crop model is treated as a black box. The development trajectory here is differentiable, so one start from the Arminda default is the search.

The agreement table and figure use the calibration site-years, the same observations the parameters were fitted to. Ears and grain number stay out, because the model does not simulate them. The eight evaluation environments can be scored only with the held-out observations. The synthetic STICS set is a separate test: it checks whether a protocol recovers parameters of STICS, not whether WOFOST can be calibrated.
